# 02｜站點小時借還量
以檔案中的時間切小時，時區尚未核對。各端分別限定在 2025 年 1 月。時長異常仍保留。

In [1]:
import sqlite3
from pathlib import Path
from zipfile import ZipFile
import pandas as pd

with ZipFile(Path('../data/raw/202501-capitalbikeshare-tripdata.zip')) as z:
    with z.open('202501-capitalbikeshare-tripdata.csv') as f:
        rides = pd.read_csv(f, dtype='string')
parts = []
for direction, time_col, id_col in [
    ('borrow', 'started_at', 'start_station_id'),
    ('return', 'ended_at', 'end_station_id'),
]:
    time = pd.to_datetime(rides[time_col], format='mixed', errors='coerce')
    station = rides[id_col].str.strip().replace('', pd.NA)
    keep = time.ge('2025-01-01') & time.lt('2025-02-01')
    print(direction, 'invalid time:', int(time.isna().sum()),
          'outside month:', int((time.notna() & ~keep).sum()))
    parts.append(pd.DataFrame({'direction': direction, 'station_id': station[keep],
                               'event_time': time[keep].dt.strftime('%Y-%m-%d %H:%M:%S.%f')}))
events = pd.concat(parts, ignore_index=True)

borrow invalid time: 0 outside month: 36
return invalid time: 0 outside month: 0


SQL 的 GROUP BY 把同一站、同一小時、同一方向放進一組。NULL 站點另外成組，代表有事件但無法歸站。

In [2]:
keys = ['station_id', 'hour', 'direction']
with sqlite3.connect(':memory:') as db:
    events.to_sql('events', db, index=False)
    sql_counts = pd.read_sql_query('''
        SELECT station_id, substr(event_time, 1, 13) || ':00:00' AS hour,
               direction, COUNT(*) AS event_count
        FROM events
        GROUP BY station_id, hour, direction
    ''', db)
py_counts = (events.assign(hour=pd.to_datetime(events.event_time).dt.floor('h')
                           .dt.strftime('%Y-%m-%d %H:%M:%S'))
             .groupby(keys, dropna=False).size().rename('event_count').reset_index())
def normalized(df):
    return df.astype({k: 'string' for k in keys}).sort_values(keys).reset_index(drop=True)
pd.testing.assert_frame_equal(normalized(sql_counts), normalized(py_counts))
assert sql_counts.event_count.sum() == len(events)
print('SQL / pandas groups matched:', len(sql_counts))
print(sql_counts.assign(station_known=sql_counts.station_id.notna())
      .groupby(['direction', 'station_known']).event_count.sum().to_string())
print('Distinct known station-hours:',
      len(sql_counts.dropna(subset=['station_id'])[['station_id','hour']].drop_duplicates()))

SQL / pandas groups matched: 200556
direction  station_known
borrow     False             88110
           True             197585
return     False             91217
           True             194514
Distinct known station-hours: 142110


這是有事件的稀疏彙總。缺少一列不代表零需求。兩套彙總共用同一份事件表，因此對帳驗證的是分組與計數，無法獨立驗證資料來源或站點營運範圍。